In [1]:
import boto3
import boto3
import duckdb
import tempfile
from datetime import datetime
import json

In [2]:
def load_master_schema():
    with open("master_schema.json", "r") as f:
        schema_dict = json.load(f)
    return schema_dict

In [3]:
def load_data_from_duckdb(valid_folder,Prefix):
    #connect to s3 bucket
    s3 = boto3.client("s3")
    bucket = "nyi-data-analytics"
    
    #connect to DuckDb
    con = duckdb.connect("bronze_staging_db.duckdb")

    master_schema = load_master_schema()

    sql_command = ",\n".join(
        f"CAST({col} AS {dtype}) AS {col}"
        for col, dtype in master_schema.items()
    )
    
    final_sql = f""" select {sql_command} , cast(ingest_timestamp as Timestamp) as ingest_timestamp , 
    row_number() over (order by created_date,unique_key) -1 as rn
    from
    staging_bronze """ 

    df = con.execute(final_sql).fetchdf()

    return df

In [4]:
def generate_folder_name(timestamp) :
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

In [5]:
def upload_df_to_s3(df, bucket_name, base_prefix, folder_name, file_name):
    
    buffer = BytesIO()

    #dataframe of raw_data
    #converts raw data to parquet and stores it in ram instead of Disk
    df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )
    
    #to bring cursor on start
    buffer.seek(0)

    #to create file path = object key
    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    
    s3.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"

In [9]:
def load_silver_to_s3(batch_size = 20000):
    
    s3 = boto3.client('s3')
    
    df = load_data_from_duckdb(1,2)

    #make batches from huge db data
    df["batch"] = df["rn"]//batch_size

    #count of batches created
    max_batch = max(df["batch"])

    #ingest timestamp for the upload 
    ingest_time_stamp= max(df["ingest_timestamp"])

    folder_name = generate_folder_name(ingest_time_stamp)

    files_created =0
    for batch in range(0,max_batch+1,1):
        #filters data for only 1 lot at a time
        batch_df = df[df["batch"]==batch]
        files_created += 1
        batch_df.drop(columns=["rn","batch"],inplace=True)

        #upload data
        batch_df.to_csv(f"part_{files_created:03d}.csv")

        load_size = len(batch_df)

        try:
            s3_path = upload_df_to_s3(
            df_batch,
            bucket_name='nyi-data-analytics',
            base_prefix='silver/full_load',
            folder_name=folder_name,
            file_name=f"part_{files_created:03d}"
            )

            s3_upload_status = f"Success: {s3_path}"

        except Exception as error:
            s3_upload_status = f"Failed: {error}"

        print("\t",load_size,"Loaded , Batch Name : ", batch , "Upload Data on S3: ", s3_upload_status)

        if batch > 0 :
            break

In [ ]:
load_silver_to_s3()

In [14]:


    print(batch)
    
    
    print(files_created, len(batch_df))
    #print(batch_df.head())
    
    
    
    print(batch , batch >2)
    

47
0
1 20000
0 False
1
2 20000
1 False
2
3 20000
2 False
3
4 20000
3 True
